In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train_transforms = transforms.Compose([

    transforms.Resize((299,299)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomVerticalFlip(),

    transforms.RandomRotation(25),

    transforms.RandomAffine(
        degrees=15,
        scale=(0.85,1.15)
    ),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3
    ),

    transforms.GaussianBlur(
        kernel_size=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_test_transforms = transforms.Compose([

    transforms.Resize((299,299)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [4]:
train_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/train",
    transform=train_transforms
)

val_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/val",
    transform=val_test_transforms
)

test_data = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/test",
    transform=val_test_transforms
)

In [5]:
train_loader = DataLoader(
    train_data,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=16,
    shuffle=False
)

class_names = train_data.classes

print(class_names)

['Brown Planthopper', 'Rice Gall Midge', 'Rice Hispa', 'Rice Leaf Folder', 'Rice Stem Borer']


In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cpu


In [7]:
model = models.inception_v3(
    weights=models.Inception_V3_Weights.DEFAULT
)

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 159MB/s] 


In [8]:
num_features = model.fc.in_features

model.fc = nn.Sequential(

    nn.Dropout(0.5),

    nn.Linear(num_features, 5)
)

model = model.to(device)

In [9]:
for param in model.parameters():
    param.requires_grad = False

In [10]:
for param in model.Mixed_7c.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

In [11]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(

    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),

    lr=1e-4,

    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

In [12]:
best_acc = 0

for epoch in range(20):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs.logits, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # VALIDATION

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs.logits,1)

            total += labels.size(0)

            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total

    print(f"🔥 Epoch {epoch+1} | Loss: {running_loss:.2f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "inceptionv3_best.pth"
        )

        print(f"✅ Best Model Saved: {best_acc:.2f}%")

    scheduler.step()

AttributeError: 'Tensor' object has no attribute 'logits'

In [13]:
# ============================================================
# 🔥 FIXED INCEPTIONV3 TRAINING LOOP
# ============================================================

best_acc = 0

epochs = 20

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # ====================================================
        # 🔥 IMPORTANT FIX
        # ====================================================

        outputs = model(images)

        # Inception sometimes returns tuple during training
        if isinstance(outputs, tuple):
            outputs = outputs[0]

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            # 🔥 FIX
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            _, preds = torch.max(outputs,1)

            total += labels.size(0)

            correct += (preds == labels).sum().item()

    val_acc = 100 * correct / total

    print(f"🔥 Epoch {epoch+1} | Loss: {running_loss:.2f} | Val Acc: {val_acc:.2f}%")

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "inceptionv3_best.pth"
        )

        print(f"✅ Best Model Saved: {best_acc:.2f}%")

🔥 Epoch 1 | Loss: 55.81 | Val Acc: 64.71%
✅ Best Model Saved: 64.71%
🔥 Epoch 2 | Loss: 48.11 | Val Acc: 72.06%
✅ Best Model Saved: 72.06%
🔥 Epoch 3 | Loss: 43.06 | Val Acc: 77.45%
✅ Best Model Saved: 77.45%
🔥 Epoch 4 | Loss: 41.32 | Val Acc: 77.94%
✅ Best Model Saved: 77.94%
🔥 Epoch 5 | Loss: 38.87 | Val Acc: 78.43%
✅ Best Model Saved: 78.43%
🔥 Epoch 6 | Loss: 38.32 | Val Acc: 77.94%
🔥 Epoch 7 | Loss: 35.98 | Val Acc: 79.90%
✅ Best Model Saved: 79.90%
🔥 Epoch 8 | Loss: 35.21 | Val Acc: 79.90%
🔥 Epoch 9 | Loss: 37.06 | Val Acc: 82.84%
✅ Best Model Saved: 82.84%
🔥 Epoch 10 | Loss: 35.77 | Val Acc: 81.86%
🔥 Epoch 11 | Loss: 35.18 | Val Acc: 81.37%
🔥 Epoch 12 | Loss: 32.58 | Val Acc: 80.88%
🔥 Epoch 13 | Loss: 32.85 | Val Acc: 82.35%
🔥 Epoch 14 | Loss: 31.64 | Val Acc: 80.39%
🔥 Epoch 15 | Loss: 31.98 | Val Acc: 81.37%
🔥 Epoch 16 | Loss: 32.29 | Val Acc: 80.39%
🔥 Epoch 17 | Loss: 32.36 | Val Acc: 79.90%
🔥 Epoch 18 | Loss: 30.85 | Val Acc: 79.90%
🔥 Epoch 19 | Loss: 32.42 | Val Acc: 80.39%
🔥 E

In [14]:
def evaluate(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            _, preds = torch.max(outputs,1)

            total += labels.size(0)

            correct += (preds == labels).sum().item()

    return 100 * correct / total


# LOAD BEST MODEL
model.load_state_dict(
    torch.load("inceptionv3_best.pth")
)

# TEST ACCURACY
test_acc = evaluate(model, test_loader)

print("🔥 Final InceptionV3 Accuracy:", test_acc)

🔥 Final InceptionV3 Accuracy: 88.9423076923077
